In [1]:
"""
This contains scratch, merging Dan's tutorial (for how to use neuralplot) with my own exploration and code for
making firing rate plots for visual presentation.
"""

"\nThis contains scratch, merging Dan's tutorial (for how to use neuralplot) with my own exploration and code for\nmaking firing rate plots for visual presentation.\n"

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from scipy.io import loadmat
import os
import numpy as np
import pandas as pd
from neuralplot import loadNeuralplot, Neuralplot
import tdt
import matplotlib.pyplot as plt

### Load data for given animal/date 
(Diego = Subj. 1, Pacho = Subj. 2)

In [10]:
# for animal, dates_list in date_animal_dict.items():
#     for date in dates_list:
#         print(f'###### Doing {animal}, {date} ######')
#         nplot = loadNeuralPlot(animal,date)
animal = 'Diego'
date = '260529'

#NOTE: Change paths to local dirs in this function >>
nplot = loadNeuralplot(animal, date)

# Testing TDT
if False:
    from tdt import read_block
    path = "/lemur2/lucas/analyses/recordings/main/MIRROR/Visual/data/Diego/260304/tdt_fixation/Diego-260304-163521"
    path = "/lemur2/lucas/analyses/recordings/main/MIRROR/Visual/data/Diego/260304/tdt_fixation/Diego-260304-163521/Diego-260304-163521"
    read_block(path)


/home/lucas/code/neuralplot/neuralplot.py:207: Warning: no tsq file found, attempting to read sev files
  session_data = read_block(fullpath,


190 [True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, False, True, True, False, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, False, True, True, False, True, True, True, True, True, True, True, False, True, True, True, True, True, False, True, False, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, False, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, Tr

AssertionError: 

### Check out data structures
- ```nplot.prettyBeh``` is the data loaded out of monkey logic

- ```nplot.prettyTdt``` is the raw data from our tdt recording system
    - This includes LFP and raw voltage signal but these are not included for sake of file sizes

- ```nplot.Dat``` is the merged dataframe, pulls information from Beh and combines it with timing precision in Tdt

- ```nplot.SpikeTimes``` is dataframe with spike times loaded from kilosort. These have been automatically clustered by ks and manually curated for 'questionable' spikes.

** All timing relating things are handled automatically to align event codes with kilosort spike times. Note that 'photodiode_time' in Tdt/Dat is the most correct timing to align with kilosort. 

** 'sample_index' is also the most accurate indexing variable, with one 'sample_index' per sample that the monkey actually saw (whether they held fixation or not). Sample index DOES NOT include cases where no sample was shown (fixation_success tracks whether he held fixation or not). In the tdt data you will see instances of this where it looks like 'fix_cue' followed immediately by 'sample_off' and 'timeout'. The fact that it says 'sample_off' (even with no 'sample_on') is just an unintended part of the monkeylogic code (i.e. it runs toggleobject(sample,'status','off') whether or not there is a toggleobject(sample,'status,'on)).

In [ ]:
nplot.prettyBeh

In [ ]:
nplot.prettyTdt

In [ ]:
nplot.Dat

In [ ]:
tdt_stim_inds = nplot.prettyTdt['stim_index'].dropna().to_numpy()
beh_stim_inds = nplot.prettyBeh['stim_index'].to_numpy()

In [ ]:
### [LT] Getting monkeylogic files for the shape kind etc
from neuralplot import lt_map_stimname_to_actual_shape_params
nplot.Dat = lt_map_stimname_to_actual_shape_params(nplot.Dat)


### Plotting!

I have included two plot functions, one to make a PSTH and one to make a raster. Functions have descriptions for use under definition. The main thing to know is what to do with ```params```

```params``` should be a dict formatted with a column name and a list of entries in that column you want to keep. The plotting functions will then filter the data to match those parameters. The PSTH plot also has a 'group_by' argument which you can use to group and average activity over different variables. Examples below.

** Note right now the plotting functions can only align to 'sample_on', the code shouldn't be too hard to write for other trial events, but wasn't sure if you would end up needing that functionality.

** All plotting functions return a dict of figs indexed by 'unit_index'. The titles have the channel number.

In [ ]:
#Plot PSTH
conditions_plot = list(range(1,50))
params = {
    'condition': conditions_plot, #plot only for these conditions
    'fixation_success_binary': [True] #only plot when fixation is successful
    #You can filter by any column/value pair here, as long as the column is present in 'Dat'
}
channel = 41
fig_dict = nplot.plotPSTH(channel, params, group_by='fixation_success_binary', window=(0.1,0.2))

In [ ]:
channel = 83


params = {
'fixation_success_binary': [True]
# 'stim_index': nplot.prettyBeh['stim_index'].to_numpy()[np.r_[500:750, 1600:1800]]
}

fig_dict = nplot.plotRaster(channel,params, window = (.4,1))


In [ ]:
nplot.spikeTimes["chan_global"].astype(int).unique()

In [ ]:
from neuralplot import lt_map_stimname_to_actual_shape_params

In [ ]:
nplot.Dat = lt_map_stimname_to_actual_shape_params(nplot.Dat)


In [ ]:
#### Extract dfallpa, and use that to map the ks units between visual and motor sessions
from neuralmonkey.scripts.analy_dfallpa_extract import extract_dfallpa_helper
from neuralmonkey.classes.population_mult import load_handsaved_wrapper, dfpa_match_chans_across_pa_each_bregion
from neuralmonkey.classes.population_mult import extract_single_pa
from neuralmonkey.metadat.analy.anova_params import params_getter_euclidian_vars
from neuralmonkey.classes.population_mult import dfpa_concatbregion_preprocess_clean_bad_channels, dfpa_concatbregion_preprocess_wrapper
from pythonlib.tools.pandastools import append_col_with_grp_index
import seaborn as sns
import os
from neuralmonkey.classes.population_mult import extract_single_pa
from neuralmonkey.analyses.state_space_good import euclidian_distance_compute_trajectories_single, euclidian_distance_compute_trajectories

animal = "Diego"
version = "trial"
combine = True
date = 260306
question = "SP_BASE_trial"


# Load a single DFallPA
DFallpa = load_handsaved_wrapper(animal, date, version=version, combine_areas=combine, 
                                    question=question)

# dfpa_concatbregion_preprocess_wrapper(DFallpa, animal, date)

# Get kilosort metadata
DFallpa[:2]
PA = DFallpa["pa"].values[0]
PA.Xlabels["chans"]

In [ ]:
# First, get dataframe that concats all the brain areas in PA

dfallpa = DFallpa[DFallpa["event"] == "03_samp"]
list_df = [pa.Xlabels["chans"] for pa in dfallpa["pa"]]
dfpachans = pd.concat(list_df, axis=0).reset_index(drop=True)
dfpachans[:2]

In [ ]:
# Given a signature of a unit, find its row in dfpachans.

def _identify_unit(dfpachans, Q, snr_final, chan_global_tdt):
    """
    Returns the one unit that matches the input signature. Fails if finds 
    number of units more than one.

    If zero, then returns None, None, None (assuming this is becuase dfpachans pruned
    units, so you can't find them)
    
    RETURNS:
    - bregion_combined, bregion, site(final unit id)
    """
    dftmp = dfpachans[(dfpachans["Q"] == Q) & (dfpachans["snr_final"] == snr_final) & (dfpachans["chan_tdt"] == int(chan_global_tdt))].reset_index(drop=True)
    
    if len(dftmp)==0:
        return None, None, None
    elif not len(dftmp)==1:
        print(len(dftmp))
        print(dftmp)
        print(Q, snr_final, chan_global_tdt)
        assert False
    else:
        # display(dftmp)
        return dftmp.loc[0, ["bregion_combined", "bregion", "chan"]].values.tolist()

Q = 0.04965934034607601
snr_final = 7.802673810960552
chan_global = 9.0
_identify_unit(dfpachans, Q, snr_final, chan_global)

In [ ]:
# Link each row of Dan's data to a specific unit number in DFallPa, using Q and snr_final, etc.
# --- write a function to dynamically do this: (PA, dandata) --> dandata(with site number as new column)
# - Then make PA from Dan's data.

def f(x):
    bregion_combined, bregion, site = _identify_unit(dfpachans, x["Q"], x["snr_final"], x["chan_global"])
    if site is None:
        return np.nan
    else:
        return int(site)
tmp = nplot.spikeTimes.apply(f, axis=1)
# Check that got all the units that you expect to get
n_na = sum(tmp.isna())
n_tot = len(tmp)

assert n_tot - n_na == len(dfpachans)
nplot.spikeTimes["site_final"] = tmp

In [ ]:
# Link each row of Dan's data to a specific unit number in DFallPa, using Q and snr_final, etc.
# --- write a function to dynamically do this: (PA, dandata) --> dandata(with site number as new column)
# - Then make PA from Dan's data.

def f(x):
    bregion_combined, bregion, site = _identify_unit(dfpachans, x["Q"], x["snr_final"], x["chan_global"])
    if bregion is None:
        return np.nan
    else:
        return bregion
tmp = nplot.spikeTimes.apply(f, axis=1)
# Check that got all the units that you expect to get
n_na = sum(tmp.isna())
n_tot = len(tmp)

assert n_tot - n_na == len(dfpachans)
nplot.spikeTimes["bregion"] = tmp

In [ ]:
nplot.spikeTimes = nplot.spikeTimes[~nplot.spikeTimes["site_final"].isna()].reset_index(drop=True)
nplot.spikeTimes["site_final"] = nplot.spikeTimes["site_final"].astype(int)

##### Helper to extract spike times

In [ ]:
# Here is all the 'raw' data. Basically the way things work now is spike times are extracted at plot time using the function 
# 'extractSpikeTimes' in the neuralplot repo. In this function you supply a list of channels and a list of times to extract 
# spikes for around a given window. 
# 
# There's a notebook in the neuralplot repo that should walk through how to load the data and get the nplot object 
# (same thing I sent before).

# ​Pancho_data.zip​​Diego_data.zip​

# For example if you wanted to get all spike times for unit 10 at stim on for trials 1-10 with a 
# +/- 0.5 second window you would input:

# ExtractSpikeTimes([10],[photodiode times trial1-10 (from nplot df)], window = (0.5,0,5)]

# Should be easy to call this function and add a column to the nplot dataframe that has spikes for a given stimulus presentation like pa.

In [ ]:
nplot.prettyTdt

In [ ]:
from neuralplot import filter_df

In [ ]:
params = {
    'fixation_success_binary': [True], #only plot when fixation is successful
    'code_type': ["sample_on"]
    #You can filter by any column/value pair here, as long as the column is present in 'Dat'
}

df = filter_df(nplot.Dat, params)

stims_in_order = sorted(set(df['stim_index']))
times_list = []
for i in stims_in_order:
    row = df.loc[df['stim_index'] == i]
    t0 = row['photodiode_time'].values[0]
    times_list.append(t0)


In [ ]:
df = filter_df(nplot.Dat, params)
df

In [ ]:
channel_list = [41]
# times_list
window = [0.5, 0.5]
dict_spike_times, index_to_channel_dict = nplot.extractSpikeTimes(channel_list, times_list, window)

In [ ]:
dict_spike_times.keys()

In [ ]:
dict_spike_times[12]

In [ ]:
index_to_channel_dict

##### LT, function to extract spike times (rewrite, like Dan's)

In [ ]:
tmp = nplot.spikeTimes[~nplot.spikeTimes["site_final"].isna()]["site_final"].astype(int).unique()
assert len(tmp) == len(set(tmp)), "somehow repeated a unit..."

In [ ]:
# Collect the times of the desired event
params = {
    'fixation_success_binary': [True], #only plot when fixation is successful
    'code_type': ["sample_on"]
    #You can filter by any column/value pair here, as long as the column is present in 'Dat'
}

dfevent = filter_df(nplot.Dat, params).reset_index(drop=True)
stim_names = dfevent["stim_name"].tolist()
event_times = dfevent["photodiode_time"].values

# Collect all the spikes for this site, (entire session, in seconds)
window = (-0.4, 1.0)
# site_final = 1001
list_df = []
_times = None
# list_site = [1001, 1002]
list_site = sorted(nplot.spikeTimes[nplot.spikeTimes["bregion"] == "PMv_l"]["site_final"].unique())
for site_final in list_site:
    print("Site: ", site_final)

    # Get single array of all spiketimes for this site
    dftmp = nplot.spikeTimes[nplot.spikeTimes["site_final"]==site_final]
    assert len(dftmp)==1
    spike_times = dftmp["times_sec_all"].values[0] # (ntimes, )

    # Get spike times.
    res = []
    for idx_trial, row in dfevent.iterrows():
        t_event = row["photodiode_time"]

        # Get event info
        stim_name = row["stim_name"]

        # Get spike timesrelative to event
        mask = (spike_times >= t_event + window[0]) & (spike_times <= t_event + window[1])
        spike_times_mask = spike_times[mask]
        st = spike_times_mask - t_event 

        # Convert spike times to rates
        SMFR_TIMEBIN = 0.01
        _SMFR_SIGMA = 0.025 # 4/20/24, # since you can always smoother further later on.
        times, fr = sn.elephant_spiketrain_to_smoothedfr(st, window[0], window[1], _SMFR_SIGMA, SMFR_TIMEBIN)
        times = np.array(times)

        res.append({
            "site_final":site_final,
            "idx_trial":idx_trial,
            "stim_name":stim_name,
            "spike_times": st,
            "smfr_rates":fr[0,:],
            "smfr_times":times[0,:],
        })

        # Sanity check that all caseas have same time base
        if _times is None:
            _times = times[0,:]
        else:
            assert np.all(_times == times[0,:])

    dfspikes = pd.DataFrame(res)
    list_df.append(dfspikes)

In [ ]:
DFSPIKES = pd.concat(list_df)

In [ ]:
# Convert to PA
sites = sorted(DFSPIKES["site_final"].unique())
trials = sorted(DFSPIKES["idx_trial"].unique())
times = _times

In [ ]:
frmat = np.stack(DFSPIKES["smfr_rates"])


In [ ]:
# Reshape
frmat = np.reshape(frmat, (len(sites), len(trials), len(times)), order="C")

# frmat2 = np.reshape(frmat, (len(trials), len(sites), len(times)), order="C")
# frmat2 = np.transpose(frmat2, (1,0,2))

In [ ]:
frmat.shape

In [ ]:
dfevent[:2]

In [ ]:
res =[]
for idx_trial, row in dfevent.iterrows():

    # Get event info
    stim_name = row["stim_name"]

    res.append({
        "idx_trial":idx_trial,
        "stim_name":row["stim_name"],
        "stim_name_index":row["stim_name_index"],
        "prim":row["prim"],
        "los_info":row["los_info"],
        "shapekind":row["shapekind"],
    })

dflab = pd.DataFrame(res)
dflab["dummy"] = "dummy"

In [ ]:
PAdan = sn._popanal_generate_from_raw(frmat, times, sites, trials=trials, df_label_trials=dflab, 
                                      list_df_label_cols_get=["idx_trial", "stim_name", "stim_name_index", "prim", "los_info", "shapekind", "dummy"])


In [ ]:
PAdan.Chans = [int(x) for x in PAdan.Chans]

In [ ]:
PAdan.Xlabels["trials"]

In [ ]:
chan = 1037
PAdan.plotwrapper_smoothed_fr_split_by_label_and_subplots(chan, "prim", ["shapekind"])


In [ ]:
for chan in PAdan.Chans:
    fig = PAdan.plotwrapper_smoothed_fr_split_by_label_and_subplots(chan, "prim", ["shapekind"])
    from pythonlib.tools.plottools import savefig 
    savefig(fig, f"/tmp/{chan}.png")
    plt.close("all")


In [ ]:
for chan in PAdan.Chans:
    fig = PAdan.plotwrapper_smoothed_fr_split_by_label_and_subplots(chan, "shapekind", ["dummy"])
    from pythonlib.tools.plottools import savefig 

    savefig(fig, f"/tmp/{chan}.png")
    plt.close("all")


In [ ]:
PAdan.plotwrappergrid_smoothed_fr_splot_neuron("prim", ["shapekind"], [1001, 1002])

In [ ]:
# TODO: Extract above for all brain regions
# Plot to match the subset of shapes done for drawing
# Do this for all dates.


In [ ]:
np.stack(dfspikes["smfr_rates"]).shape

In [ ]:
# Convert to PA


In [ ]:
from neuralmonkey.classes.session import Session
sn = Session([], [], [], ACTUALLY_BAREBONES_LOADING=True)

In [ ]:
for _, ro

In [ ]:
st = dfspikes["spike_times"].values[0]

In [ ]:
# st = self.elephant_spiketrain_from_values(dat["spike_times"], dat["time_on"], dat["time_off"])
# list_spiketrain = [st]
# frate = instantaneous_rate(list_spiketrain, sampling_period=sampling_period*s, 
#     kernel=GaussianKernel(gaussian_sigma*s))

# PA = self._popanal_generate_from_raw(frate.T.magnitude, frate.times, list_sites, df_label_trials=None)


In [ ]:
SMFR_TIMEBIN = 0.01
_SMFR_SIGMA = 0.025 # 4/20/24, # since you can always smoother further later on.
times, fr = sn.elephant_spiketrain_to_smoothedfr(st, window[0], window[1], _SMFR_SIGMA, SMFR_TIMEBIN)
times = np.array(times)

In [ ]:
nplot.spikeTimes["site_final"].unique()

### Additional functions in quad.py

```filter_df``` = function to filter df given dict (structured like params above). Filters df by column/value pairs and returns the filter df. Useful for plotting.

```group_and_average``` = useful function for plotting, will take dat and group by unique values in a particular column and then average over 'spike_counts'. Right now, for example you could make one psth for each unique stimulus. If you wanted to do something like regular vs irregular though, you could edit this function to be more flexible to that.

```getChannelNumOrRegionName``` = A flexible function for getting the channels associated with a region or the region associated with a channel. You can look at the bottom of the file 'MAP_CHANNEL_TO_REGION' is a dict with all this info.